In [2]:
%%sql
SELECT
    p.age,
    p."capital-gain",
    p."capital-loss",
    p."hours-per-week",
    p.income,
    d."marital-status",
    d.relationship,
    d.race,
    d.sex,
    d."native-country",
    e.education,
    o.workclass,
    o.occupation
FROM person p
JOIN demographics d ON p.demographics_id = d.id
JOIN education    e ON p.education_id    = e.id
JOIN occupation   o ON p.occupation_id   = o.id;

,age,capital-gain,capital-loss,hours-per-week,income,marital-status,relationship,race,sex,native-country,education,workclass,occupation
0,39,2174,0,NaN,<=50K,Never-married,Not-in-family,White,Male,United-States,Bachelors,State-gov,Adm-clerical
1,50,0,0,13.0,<=50K,Married-civ-spouse,Husband,White,Male,United-States,Bachelors,Self-emp-not-inc,Exec-managerial
2,38,0,0,40.0,<=50K,Divorced,Not-in-family,White,Male,United-States,HS-grad,Private,Handlers-cleaners
3,53,0,0,40.0,<=50K,Married-civ-spouse,Husband,Black,Male,United-States,11th,Private,Handlers-cleaners
4,28,0,0,NaN,<=50K,Married-civ-spouse,Wife,Black,Female,Cuba,Bachelors,Private,Prof-specialty
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,0,0,38.0,<=50K,Married-civ-spouse,Wife,White,Female,United-States,Assoc-acdm,Private,Tech-support
32557,40,0,0,NaN,>50K,Married-civ-spouse,Husband,White,Male,United-States,HS-grad,Private,Machine-op-inspct
32558,58,0,0,40.0,<=50K,Widowed,Unmarried,White,Female,United-States,HS-grad,Private,Adm-clerical
32559,22,0,0,20.0,<=50K,Never-married,Own-child,White,Male,United-States,HS-grad,Private,Adm-clerical


In [3]:
df_raw

,age,capital-gain,capital-loss,hours-per-week,income,marital-status,relationship,race,sex,native-country,education,workclass,occupation
0,39,2174,0,NaN,<=50K,Never-married,Not-in-family,White,Male,United-States,Bachelors,State-gov,Adm-clerical
1,50,0,0,13.0,<=50K,Married-civ-spouse,Husband,White,Male,United-States,Bachelors,Self-emp-not-inc,Exec-managerial
2,38,0,0,40.0,<=50K,Divorced,Not-in-family,White,Male,United-States,HS-grad,Private,Handlers-cleaners
3,53,0,0,40.0,<=50K,Married-civ-spouse,Husband,Black,Male,United-States,11th,Private,Handlers-cleaners
4,28,0,0,NaN,<=50K,Married-civ-spouse,Wife,Black,Female,Cuba,Bachelors,Private,Prof-specialty
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,0,0,38.0,<=50K,Married-civ-spouse,Wife,White,Female,United-States,Assoc-acdm,Private,Tech-support
32557,40,0,0,NaN,>50K,Married-civ-spouse,Husband,White,Male,United-States,HS-grad,Private,Machine-op-inspct
32558,58,0,0,40.0,<=50K,Widowed,Unmarried,White,Female,United-States,HS-grad,Private,Adm-clerical
32559,22,0,0,20.0,<=50K,Never-married,Own-child,White,Male,United-States,HS-grad,Private,Adm-clerical


In [4]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import json, yaml

TARGET_COL = "income"
df_features = df_raw.drop(columns=[TARGET_COL])
df_clean = df_raw.copy()

issues = []

for col in df_features.columns:
    n = df_features[col].isnull().sum()
    if n > 0:
        pct = n / len(df_features) * 100
        issues.append({
            "type": "missingness", "column": col,
            "detail": f"{n:,} missing ({pct:.1f}%)",
            "severity": "high" if pct > 30 else "medium" if pct > 5 else "low"
        })

for col in df_features.select_dtypes(include=[np.number]).columns:
    s = df_features[col].dropna()
    if len(s) < 10: continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_iqr = int(((s < q1 - 3*iqr) | (s > q3 + 3*iqr)).sum())
    n_z   = int((np.abs(stats.zscore(s)) > 3).sum())
    if n_iqr > 0 or n_z > 0:
        issues.append({
            "type": "outlier", "column": col,
            "detail": f"IQR outliers: {n_iqr:,} | z-score outliers: {n_z:,}",
            "severity": "high" if max(n_iqr, n_z) / len(s) > 0.05 else "low"
        })

n_dup = int(df_features.duplicated().sum())
if n_dup > 0:
    issues.append({
        "type": "duplicates", "column": "ALL",
        "detail": f"{n_dup:,} exact duplicate rows ({n_dup/len(df_features)*100:.1f}%)",
        "severity": "high" if n_dup / len(df_features) > 0.01 else "low"
    })

sev_order = {"high": 0, "medium": 1, "low": 2}
issues_df = (pd.DataFrame(issues)
    .assign(_o=lambda d: d["severity"].map(sev_order))
    .sort_values("_o").drop(columns="_o")
    .reset_index(drop=True))
issues_df.index += 1

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols | target: {TARGET_COL}")
print(f"Issues: {len(issues)}  (high: {sum(1 for i in issues if i['severity']=='high')}  "
      f"medium: {sum(1 for i in issues if i['severity']=='medium')}  "
      f"low: {sum(1 for i in issues if i['severity']=='low')})")
print(f"\n{issues_df.to_string()}")

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("configs").mkdir(exist_ok=True)
json.dump({"shape": list(df_raw.shape),
           "dtypes": df_raw.dtypes.astype(str).to_dict(),
           "missing": df_raw.isnull().sum().to_dict(),
           "missing_pct": (df_raw.isnull().mean() * 100).round(2).to_dict(),
           "nunique": df_raw.nunique().to_dict()},
          open("data/raw/profile.json", "w"), default=str, indent=2)
yaml.dump({"target_column": TARGET_COL,
           "n_rows": int(df_raw.shape[0]),
           "n_cols": int(df_raw.shape[1])},
          open("configs/cleaning_config.yaml", "w"))

Shape: 32,561 rows x 13 cols | target: income
Issues: 7  (high: 3  medium: 1  low: 3)

          type          column                                         detail severity
1      outlier    capital-gain    IQR outliers: 2,712 | z-score outliers: 215     high
2      outlier  hours-per-week    IQR outliers: 3,826 | z-score outliers: 363     high
3   duplicates             ALL             3,491 exact duplicate rows (10.7%)     high
4  missingness  hours-per-week                          4,784 missing (14.7%)   medium
5  missingness       workclass                           1,320 missing (4.1%)      low
6      outlier             age        IQR outliers: 0 | z-score outliers: 121      low
7      outlier    capital-loss  IQR outliers: 1,519 | z-score outliers: 1,470      low


### Outliers: `capital-gain`
- Outliers: 2,712 (8.3%) IQR / 215 z-score  |  Interpretation: genuine extremes
- Fix: `log1p` transform  |  Affected: all 32,561 rows (values reshaped, none dropped)
- Rationale: distribution is zero-inflated with top-coding at $99,999; log1p compresses the tail while preserving zeros.

In [5]:
# Outliers: capital-gain
print(f"Before: min={df_clean['capital-gain'].min():.4g}, max={df_clean['capital-gain'].max():.4g}, skew={df_clean['capital-gain'].skew():.2f}")
df_clean["capital-gain"] = np.log1p(df_clean["capital-gain"])
print(f"After:  min={df_clean['capital-gain'].min():.4g}, max={df_clean['capital-gain'].max():.4g}, skew={df_clean['capital-gain'].skew():.2f}")

Before: min=0, max=1e+05, skew=11.95
After:  min=0, max=11.51, skew=3.10


### Outliers: `hours-per-week`
- Outliers: 3,826 IQR / 363 z-score  |  Interpretation: mixed
  - 64 rows at hpw=99 → top-coded, set to NaN (imputed in next step)
  - Remainder → genuine extremes, winsorize at 1st/99th percentile
- Rationale: 99 is a reporting ceiling artifact (cf. capital-gain at $99,999); the rest reflects real part-time / long-hours work.

In [6]:
# Outliers: hours-per-week
n_topcoded = int((df_clean["hours-per-week"] == 99).sum())
df_clean.loc[df_clean["hours-per-week"] == 99, "hours-per-week"] = np.nan
print(f"Top-coded (=99) -> NaN: {n_topcoded} rows")

lo, hi = df_clean["hours-per-week"].quantile([0.01, 0.99])
print(f"Winsorize bounds: 1st pct={lo:.0f}, 99th pct={hi:.0f}")
print(f"Before: min={df_clean['hours-per-week'].min():.0f}, max={df_clean['hours-per-week'].max():.0f}")
df_clean["hours-per-week"] = df_clean["hours-per-week"].clip(lower=lo, upper=hi)
print(f"After:  min={df_clean['hours-per-week'].min():.0f}, max={df_clean['hours-per-week'].max():.0f}, missing={df_clean['hours-per-week'].isnull().sum()}")

Top-coded (=99) -> NaN: 64 rows
Winsorize bounds: 1st pct=8, 99th pct=75
Before: min=1, max=98
After:  min=8, max=75, missing=4848


### Duplicates
- Exact duplicates: 3,491 rows (10.7%) measured on `df_raw` features
- Fix: Drop duplicates, keep first occurrence
- Note: effective count on current `df_clean` may differ slightly since `hours-per-week` was modified (winsorize + NaN) in the previous step.

In [7]:
# Duplicates
print(f"Before: {len(df_clean):,} rows | duplicates now: {df_clean.duplicated().sum():,}")
df_clean = df_clean.drop_duplicates(keep="first").reset_index(drop=True)
print(f"After:  {len(df_clean):,} rows")

Before: 32,561 rows | duplicates now: 2,984
After:  29,577 rows


### Missingness: `hours-per-week`
- Missing: ~4,800 (~16%)  |  Mechanism: **MAR** (with possible MNAR component)
  - Strongly tied to income (>50K: 30% missing vs ≤50K: 10%) and occupation (Exec-managerial / Prof-specialty / Self-emp-inc most missing; service jobs least)
  - Numeric correlations weak (|r| ≤ 0.06) — signal lives in categoricals
- Fix: **Iterative imputation** + **missingness indicator column** (`hours-per-week_was_missing`)
  - Iterative imputation conditions on observed variables (correct for MAR)
  - Indicator preserves the "didn't report" signal in case the >50K group has hidden MNAR
- Imputed values clipped to the winsorize bounds [8, 75] and rounded to integers.